# Agent 第 13 课：Bedrock Agents（托管版）第一印象

上一课我们还在自己写 while 循环 + tool dispatch。这一课把**整个循环交给 AWS**。

Bedrock Agents 是 AWS 的托管 agent 服务：你给它 foundation model + instruction + 几个 Action Group（工具），它在后台跑 ReAct 循环、存 session、做 trace log、挂 Guardrails。

学完这节你能回答：
1. 一个 Bedrock Agent 是由哪几块 resource 组成的
2. 为什么要分 "DRAFT version" / "alias" / "published version"
3. 什么情况下该用托管 Agent、什么情况下该继续自己写

## 1. Bedrock Agent 的组成

一个 Agent = 下面这些 resource 的组合：

```
Agent
├── foundationModel       (e.g. claude-sonnet-4-6 的 modelId)
├── instruction           (system prompt，最重要)
├── idleSessionTTLInSeconds
├── Agent Version (数字)  ← 不可变，一次发布一个
│   └── Agent Alias      ← 生产流量指向这个（指向某个 version）
├── Action Group (可多个)
│   ├── schema (OpenAPI 3.0 或 function schema)
│   └── Lambda ARN       ← 实际执行工具调用的 Lambda
├── Knowledge Base associations (可多个)
└── Guardrail ID         (可选，对应 Bedrock Guardrails)
```

**对照 Phase 1**：
- instruction = 你的 system prompt
- Action Group = 一组工具 + 它们的实现（Lambda 代替你的 Python 函数）
- Knowledge Base = 第 5 课的长期记忆 / RAG
- Guardrail = 第 10 课的安全层
- AWS 跑的那个循环 = 第 3 课你手写的 while 循环

## 2. 版本与 Alias —— 别绕进去

这个很多人第一次用会晕：

- 你**编辑**的永远是 `DRAFT` 版本
- 调完满意了，调 `PrepareAgent` + `CreateAgentVersion` → 生成一个**不可变版本**（Version 1, Version 2…）
- 生产流量不直接打 version，打 **Alias**；alias 指向某个 version
- 要发新版就把 alias 重定向到 Version N+1 → 秒级发布，要回滚就指回 N

类似 Lambda alias。**生产代码里永远用 alias，不要 hard-code version**。

## 3. 两种使用方式

**方式 A：Console 点出来**（第一次玩用这个，5 分钟搞定）
1. Bedrock Console → Agents → Create
2. 选 model、写 instruction
3. 加 Action Group（上传 OpenAPI + 选 Lambda）
4. Prepare → Test 窗口里直接聊
5. 满意了 Create Alias

**方式 B：boto3 全程代码创建**（CI/CD 要用）
用 `bedrock-agent` 客户端（**注意**：不是 `bedrock-runtime`）创建资源；调用 agent 用 `bedrock-agent-runtime` 客户端。

> 几个 client 要区分清楚：
> - `bedrock`            —— 管 model access（列模型、申请 access）
> - `bedrock-runtime`    —— 直接调模型（上一课的 Converse）
> - `bedrock-agent`      —— 管 Agent / KB / Guardrail 这些 resource（create / update）
> - `bedrock-agent-runtime` —— **调用**已经建好的 Agent / KB

In [ ]:
import boto3, os, time, json, uuid
os.environ.setdefault("AWS_REGION", "us-west-2")

agent_client = boto3.client("bedrock-agent")            # control plane
runtime      = boto3.client("bedrock-agent-runtime")    # invoke
sts          = boto3.client("sts")
ACCOUNT = sts.get_caller_identity()["Account"]
print("account =", ACCOUNT)

## 4. 最小 Agent：创建 → Prepare → 调用

下面这段**能跑完**，前提你账户里有一个 IAM role 让 Bedrock Agent 能用。通常叫 `AmazonBedrockExecutionRoleForAgents_*`，trust policy 允许 `bedrock.amazonaws.com` assume。

没有的话去 IAM Console 建一个，trust policy：
```json
{"Version":"2012-10-17","Statement":[{"Effect":"Allow",
 "Principal":{"Service":"bedrock.amazonaws.com"},"Action":"sts:AssumeRole"}]}
```
权限给 `AmazonBedrockFullAccess`（DEMO 用；生产要收紧）。

In [ ]:
# 把下面这一行改成你的 role ARN 再运行
AGENT_ROLE_ARN = f"arn:aws:iam::{ACCOUNT}:role/AmazonBedrockExecutionRoleForAgents_Demo"
FOUNDATION_MODEL = "anthropic.claude-sonnet-4-6-20260101-v1:0"  # 按账户可见模型改

# 1) 创建 agent（初始是 DRAFT）
resp = agent_client.create_agent(
    agentName=f"demo-agent-{uuid.uuid4().hex[:6]}",
    agentResourceRoleArn=AGENT_ROLE_ARN,
    foundationModel=FOUNDATION_MODEL,
    instruction=(
        "You are a simple assistant that answers factual questions concisely."
        " If you don't know, say so. Never make up numbers."
    ),
    idleSessionTTLInSeconds=600,
    description="Lesson 13 demo",
)
AGENT_ID = resp["agent"]["agentId"]
print("agentId =", AGENT_ID, "status =", resp["agent"]["agentStatus"])

In [ ]:
# 2) Prepare：把 DRAFT 编译成可调用状态
agent_client.prepare_agent(agentId=AGENT_ID)

# 轮询到 PREPARED
while True:
    s = agent_client.get_agent(agentId=AGENT_ID)["agent"]["agentStatus"]
    print("status =", s)
    if s == "PREPARED": break
    if s in ("FAILED", "DELETING"): raise RuntimeError(s)
    time.sleep(3)

In [ ]:
# 3) 创建一个 Alias 指向 DRAFT（生产中应指向 Version N；demo 简化）
alias_resp = agent_client.create_agent_alias(
    agentId=AGENT_ID,
    agentAliasName="dev",
    description="dev alias pointing to latest DRAFT",
)
AGENT_ALIAS_ID = alias_resp["agentAlias"]["agentAliasId"]
print("aliasId =", AGENT_ALIAS_ID)

In [ ]:
# 4) 调用 agent：invoke_agent 是**流式**响应，要迭代 event stream
def invoke(text, session_id=None):
    session_id = session_id or str(uuid.uuid4())
    stream = runtime.invoke_agent(
        agentId=AGENT_ID,
        agentAliasId=AGENT_ALIAS_ID,
        sessionId=session_id,
        inputText=text,
    )
    parts = []
    for event in stream["completion"]:
        if "chunk" in event:
            parts.append(event["chunk"]["bytes"].decode())
        # trace 事件在 event["trace"] 里——生产调试全靠它，见下一 cell
    return session_id, "".join(parts)

sid, answer = invoke("用一句话解释什么是 AWS Bedrock Agent。")
print(answer)

## 5. Trace：托管 agent 的"调试显微镜"

本地手写 agent 你自己 print thought/action。托管 agent 你 print 不到——但 AWS 给了一个 `trace` 事件流，把 agent 内部每一步（routing、orchestration、知识库查询、tool 调用、model 输入输出）都吐出来。

开启方式：`invoke_agent(enableTrace=True, ...)`，然后在 event stream 里看 `event["trace"]`。

**生产上一定要开 trace 并写到 CloudWatch / S3**——否则线上 agent 胡说八道你束手无策。

In [ ]:
def invoke_with_trace(text):
    sid = str(uuid.uuid4())
    stream = runtime.invoke_agent(
        agentId=AGENT_ID, agentAliasId=AGENT_ALIAS_ID,
        sessionId=sid, inputText=text, enableTrace=True,
    )
    answer_parts = []
    for event in stream["completion"]:
        if "chunk" in event:
            answer_parts.append(event["chunk"]["bytes"].decode())
        elif "trace" in event:
            t = event["trace"]["trace"]
            ttype = next(iter(t))
            print(f"  [trace:{ttype}]  {json.dumps(t[ttype], default=str)[:200]}...")
    return "".join(answer_parts)

print(invoke_with_trace("美国人口大约多少？"))

## 6. 什么时候托管、什么时候继续自己写

**用 Bedrock Agents 很划算的场景**：
- 你要的就是 "LLM + 工具 + RAG" 的标准组合
- 团队希望把运维（session 存储、retry、trace、日志）交出去
- 企业合规要求：Bedrock 有 VPC 链路、KMS、Guardrails 打包
- 需要快速多版本 + alias 切换发布流程

**继续手写（Phase 1 那套）更合适**：
- 你有**非标准的循环**（嵌套 agent、高度定制的 planner、特殊 reflection 策略）
- 需要极致优化 token（自己拼 prompt / 自定义 memory 压缩）
- 不想被 AWS 抽象锁死（portable 到其它云）
- 做研究 / 原型，要最大灵活度

**一个常见的混搭**：核心流程用 Bedrock Agent，特殊子任务用 `bedrock-runtime` 的 Converse 自己写。两个 client 能共存。

---

## 7. 清理 resources（demo 结尾一定要做）

In [ ]:
# 清理：删 alias -> 删 agent
try:
    agent_client.delete_agent_alias(agentId=AGENT_ID, agentAliasId=AGENT_ALIAS_ID)
    print("deleted alias")
except Exception as e:
    print("alias delete skipped:", e)

try:
    agent_client.delete_agent(agentId=AGENT_ID, skipResourceInUseCheck=True)
    print("deleted agent")
except Exception as e:
    print("agent delete skipped:", e)

## 8. 工程直觉

1. **Alias 是生产唯一入口**。hard-code version = 回滚慢。
2. **Trace 默认开启**，不然线上 debug 没法干。
3. **两套 client 别搞混**：`bedrock-agent` 建资源，`bedrock-agent-runtime` 调用。
4. **IAM role trust 出错是第一大踩坑**。Agent role 必须信任 `bedrock.amazonaws.com`。
5. **下一课**：Action Groups —— 把真实工具（Lambda）挂上去，让 agent 真的能动手。